In [1]:
!pip install sqlalchemy

In [2]:
!pip install pymysql

In [1]:
import pandas as pd
import numpy as np
import pymysql
from sqlalchemy import create_engine

In [4]:
# 1.Establish a connection between Python and the Sakila database.

In [2]:
import getpass  
password = getpass.getpass()

 ········


In [4]:
connection = engine.connect()

In [8]:
%pip install mysql-connector-python

Note: you may need to restart the kernel to use updated packages.


In [5]:
import mysql.connector

In [ ]:
conn = mysql.connector.connect(host='localhost',
                        user='root',
                        passwd=password, database = "sakila")

In [ ]:
cursor = conn.cursor()

In [ ]:
# Execute as many queries as you want
cursor.execute("SELECT * FROM actor LIMIT 5")
print(cursor.fetchall())

cursor.execute("SELECT COUNT(*) FROM film")
print(cursor.fetchone())

cursor.execute("SELECT first_name FROM actor LIMIT 3")
print(cursor.fetchall())

In [ ]:
# Close everything when you're done
cursor.close()
conn.close()

In [ ]:
#Write a Python function called rentals_month that retrieves rental data for a given month and year (passed as parameters) from the 
#Sakila database as a Pandas DataFrame. The function should take in three parameters:

#engine: an object representing the database connection engine to be used to establish a connection to the Sakila database.
#month: an integer representing the month for which rental data is to be retrieved.
#year: an integer representing the year for which rental data is to be retrieved.

#The function should execute a SQL query to retrieve the rental data for the specified month and year from 
#the rental table in the Sakila database, and return it as a pandas DataFrame.

In [ ]:
def rentals_month(engine, month, year=2005):

    query = "SELECT * FROM rental WHERE MONTH(rental_date) = "+str(month)+" AND YEAR(rental_date) = "+str(year)
    
    with engine.connect() as connection:    
        query = text(query)
        result = connection.execute(query)
        df = pd.DataFrame(result.all())
    
    return df

In [ ]:
may = rentals_month(engine, 5, 2005)
may.head(1)

In [ ]:
#Develop a Python function called rental_count_month that takes the DataFrame provided by rentals_month as input along with the month and year and 
#returns a new DataFrame containing the number of rentals made by each customer_id during the selected month and year.
#The function should also include the month and year as parameters and use them to name the new column according to the month and year, for example, 
#if the input month is 05 and the year is 2005, the column name should be "rentals_05_2005".

In [ ]:
def rental_count_month(df, month, year=2005):

    rentals_by_customer = df.loc[(df['rental_date'].dt.month == month) & (df['rental_date'].dt.year == year)]\
                          .groupby('customer_id')['rental_id'].count().reset_index()

    rentals_by_customer = rentals_by_customer.rename(columns={'rental_id': f'rentals_{month}_{year}'})

    return rentals_by_customer

In [ ]:
customer_rentals_may = rental_count_month(may, 5)
customer_rentals_may

In [ ]:
#4.Create a Python function called compare_rentals that takes two DataFrames as input containing the number of rentals made by each customer in 
#different months and years. The function should return a combined DataFrame with a new 'difference' column, which is the difference between the 
#number of rentals in the two months.

In [ ]:
def compare_rentals(df1, df2):

    merged_df = pd.merge(df1, df2, on='customer_id')

    merged_df['difference'] = merged_df[df1.columns[1]] - merged_df[df2.columns[1]]

    return merged_df

In [ ]:
june = rentals_month(engine,6)
customer_rentals_june = rental_count_month(june, 6)
customer_rentals_june

comparative = compare_rentals(customer_rentals_may, customer_rentals_june)
comparative